# 01 · Mass spectra and the raw data

Following the tutorial notebook [`01_mass_spectra_and_data_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level1/01_mass_spectra_and_data_student.ipynb).

**Kernel:** `cajal-lipidomics`

**What I'm doing here**
- Pull the control and pregnant sections from METASPACE (CoreMetabolome v3, FDR 0.1)
- Mask to tissue, keep ions found in both sections, build the pixels x ions matrix
- Attach the provided Allen registration (CCF coords + region) to every pixel
- Save `01_raw.h5ad`, then make the first lipid maps

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages I've built so far.
                      # If a line starts with "--", that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from my clone in external/).
# Rule from the course: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
cl.style.set_style()               # the course's figure defaults, so plots look clean and consistent

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

## load

In [ ]:
# ---- inputs for this notebook ------------------------------------------------------------
# NB01 is the start of the chain, so there's no previous stage to load. Instead I pull the raw
# MALDI images from METASPACE (later in the notebook) and combine them with two things from the
# tutorial's data bundle:

# 1. The registration table: for every tissue pixel of both sections, its Allen CCF coordinates
#    (xccf, yccf, zccf, in mm), the Allen region acronym, and that region's official color.
#    This is the ABBA output the course ships pre-computed. I'll join it onto my pixels by
#    (SectionID, x, y).
registration = pd.read_parquet(paths.tutorial_data("provided", "registration_ccf.parquet"))

# 2. The tissue masks: boolean images, True where there's brain, False on empty glass.
#    I keep only the True pixels so the background doesn't pollute anything downstream.
#    One folder per section (control vs pregnant).
mask_dir_control  = paths.tutorial_data("masks", "BrainAtlas", "Control_Brains", "female")
mask_dir_pregnant = paths.tutorial_data("masks", "PREGNANT")

# Quick look: rows = tissue pixels across both sections, columns = coordinates + region info
print("registration table:", registration.shape)
print("mask folders found:", mask_dir_control.exists(), mask_dir_pregnant.exists())

# ---- output -----------------------------------------------------------------------------------
# When I've built the AnnData (pixels x ions + obs + var), I save it as stage "raw":
#   raw.write_h5ad(paths.stage("raw"))      ->  data/derived/01_raw.h5ad

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time